# Análisis exploratorio de datos (EDA)



El Análisis Exploratorio de Datos (EDA) corresponde a la etapa en la que se examinan las características, distribución, variabilidad y relaciones de los datos antes de aplicar técnicas de modelamiento.

En este proyecto, el EDA tiene como objetivo comprender el comportamiento de los indicadores financieros de las entidades bancarias privadas a lo largo del período analizado, identificar diferencias entre instituciones, detectar posibles valores atípicos y evaluar relaciones de dependencia o redundancia entre las variables.

El análisis se desarrolla mediante estadística descriptiva, análisis de evolución temporal, mapas de calor, diagramas de caja y análisis de correlación.

Los resultados del EDA se utilizan para sustentar las decisiones de preparación de la base que posteriormente será utilizada en el análisis de reducción de dimensionalidad mediante PCA y en la conformación de perfiles mediante clustering.

Es importante señalar que la identificación de valores atípicos no implica automáticamente su eliminación. Las observaciones extremas pueden corresponder a características financieras reales de determinadas entidades, por lo que su tratamiento requiere una evaluación individual que realizaremos a continuación.

## Carga de las librerías y de los módulos

In [ ]:
import os
import matplotlib
from matplotlib.colors import ListedColormap, BoundaryNorm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import os
import re

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import importlib

from src import eda_process as eda


In [ ]:
importlib.reload(eda)

In [ ]:
from pathlib import Path

RAIZ_PROYECTO = "."

In [ ]:
# Se convierte la ruta recibida desde Papermill a un objeto Path,
# permitiendo construir las rutas del proyecto de forma independiente
# del sistema operativo y del directorio desde el cual se ejecute el notebook.
RAIZ_PROYECTO = Path(RAIZ_PROYECTO)

print(f"Raíz del proyecto: {RAIZ_PROYECTO}")

In [ ]:
# Se agrega la raíz del proyecto al path de Python para permitir
# la importación de los módulos almacenados en la carpeta src.
sys.path.append(str(Path(RAIZ_PROYECTO)))

In [ ]:
# Se construye la ruta de la base generada en la etapa de preprocesamiento.
ruta_base = (
    RAIZ_PROYECTO
    / "data_processed"
    / "base_preprocesada1.xlsx"
)

# Se carga la base preprocesada que será utilizada como insumo del EDA.
base_final = pd.read_excel(ruta_base)

print(f"Base cargada desde: {ruta_base}")

In [ ]:
base_final = base_final.copy()

In [ ]:
base_final

In [ ]:
# Se identifican las variables cuantitativas que serán consideradas
# en el análisis estadístico y gráfico.

columnas_numericas = eda.obtener_columnas_numericas(base_final)
print("Cantidad de variables:", len(columnas_numericas))

In [ ]:
estadisticas = eda.estadistica_descriptiva(base_final)

estadisticas

Se simplifica el nombre del indicador `( PATRIMONIO + RESULTADOS ) / ACTIVOS INMOVILIZADOS NETOS (3) (6)` a `SUFICIENCIA PATRIMONIAL` para facilitar su lectura en tablas, gráficos y análisis posteriores.

In [ ]:
base_final.rename(
    columns={"( PATRIMONIO + RESULTADOS ) / ACTIVOS INMOVILIZADOS NETOS (3) (6)":"SUFICIENCIA PATRIMONIAL"}, inplace=True)

## Interpretación de indicadores financieros y sus límites regulatorios

En esta sección se analizan los indicadores financieros generales extraídos de la hoja `INDICADORES` de los boletines mensuales de la Superintendencia de Bancos.  
Estos indicadores se agrupan de acuerdo con la dirección
deseable de su comportamiento financiero. Esta clasificación permite
interpretar los resultados considerando que, dependiendo del indicador,
un valor elevado puede representar un mejor desempeño o, por el contrario,
un mayor nivel de riesgo.

Los límites y referencias regulatorias se presentan como elementos de
contexto para interpretar los resultados, diferenciándolos de los
criterios estadísticos utilizados posteriormente.

## Indicadores Financieros



La siguiente tabla presenta el significado, las principales referencias de interpretación y la dirección deseable de los indicadores financieros utilizados en el análisis. Es importante distinguir entre los límites establecidos por la normativa y las referencias financieras o matemáticas. Cuando no existe un límite regulatorio general, se utiliza la dirección deseable del indicador para facilitar su interpretación durante el EDA.

| Variable | Significado | Límite / referencia | Interpretación | Dirección deseable |
|:---|:---|:---|:---|:---|
| **Suficiencia patrimonial** | Mide la capacidad del patrimonio técnico para respaldar los activos y contingentes ponderados por riesgo y absorber pérdidas. | **≥ 9%** | Valores inferiores al 9% representan un incumplimiento del mínimo regulatorio. | **Mayor es mejor** |
| **Índice de capitalización neto** | Mide la relación del patrimonio frente al total de activos y contingentes, reflejando el nivel de capitalización de la entidad. | **Referencia: ≥ 4%** para la relación de patrimonio técnico frente a activos totales y contingentes. | Un mayor nivel de capitalización implica una mayor capacidad para absorber pérdidas. | **Mayor es mejor** |
| **Activos improductivos netos / Total activos** | Mide qué proporción de los activos no genera rendimientos, neta de provisiones. | Sin límite regulatorio general identificado. | Una proporción elevada representa una mayor cantidad de recursos comprometidos en activos que no generan rentabilidad. | **Menor es mejor** |
| **Activos productivos / Total activos** | Mide la proporción de los activos de la entidad que genera ingresos o rendimiento. | **Rango matemático: 0--100%**. | Una mayor proporción indica una mayor utilización productiva de los activos. | **Mayor es mejor** |
| **Activos productivos / Pasivos con costo** | Mide la capacidad de los activos productivos para cubrir los pasivos que generan costo financiero. | **100% = cobertura 1 a 1**. Puede superar el 100%. | Valores superiores al 100% indican que los activos productivos superan a los pasivos con costo. | **Mayor es mejor** |
| **Morosidad de la cartera total** | Mide el porcentaje de la cartera que presenta problemas de pago respecto de la cartera bruta. | **0%** es el mínimo matemático. Sin máximo regulatorio general identificado. | Una menor tasa de morosidad indica una mejor calidad de la cartera. | **Menor es mejor** |
| **Morosidad de la cartera de consumo** | Mide el porcentaje de la cartera de consumo que se encuentra en situación de morosidad. | **0%** es el mínimo matemático. Sin máximo regulatorio general identificado. | Una menor tasa indica un menor deterioro de la cartera de consumo. | **Menor es mejor** |
| **Morosidad de la cartera de microcrédito** | Mide el porcentaje de la cartera de microcrédito que presenta morosidad. | **0%** es el mínimo matemático. Sin máximo regulatorio general identificado. | Una menor tasa indica una mejor calidad de la cartera de microcrédito. | **Menor es mejor** |
| **Morosidad de la cartera inmobiliaria** | Mide el porcentaje de la cartera inmobiliaria que presenta morosidad. | **0%** es el mínimo matemático. Sin máximo regulatorio general identificado. | Una menor tasa representa un menor deterioro de la cartera inmobiliaria. | **Menor es mejor** |
| **Cobertura de la cartera problemática** | Mide cuánto de la cartera problemática se encuentra respaldado por provisiones. | **100% = cobertura completa**. Puede superar el 100%. | Valores superiores al 100% indican que las provisiones superan la cartera problemática. | **Mayor es mejor** |
| **Cobertura de la cartera refinanciada** | Mide el nivel de provisiones respecto de la cartera refinanciada. | **100% = cobertura completa**. | Una mayor cobertura implica un mayor respaldo mediante provisiones. | **Mayor es mejor** |
| **Cobertura de la cartera reestructurada** | Mide el nivel de provisiones respecto de la cartera reestructurada. | **100% = cobertura completa**. Puede superar el 100%. | Una mayor cobertura representa una mayor protección frente al riesgo de esta cartera. | **Mayor es mejor** |
| **Gastos de operación estimados / Total activo promedio** | Mide el peso de los gastos operativos respecto de los activos promedio. | Sin límite regulatorio general identificado. | Un porcentaje elevado indica un mayor consumo de recursos para la operación de la entidad. | **Menor es mejor** |
| **Gastos de operación / Margen financiero** | Mide cuánto del margen financiero es absorbido por los gastos de operación. | **100% = los gastos absorben todo el margen financiero**. | Valores superiores al 100% indican que los gastos operativos superan al margen financiero. | **Menor es mejor** |
| **Gastos de personal estimados / Activo promedio** | Mide el peso de los gastos de personal respecto de los activos promedio. | Sin límite regulatorio general identificado. | Un porcentaje elevado indica un mayor peso relativo de los gastos de personal. | **Menor es mejor** |
| **ROE (Resultados del ejercicio / Patrimonio promedio)** | Mide la rentabilidad obtenida sobre el patrimonio promedio. | Sin máximo regulatorio general identificado. **0% = ausencia de rentabilidad; <0% = pérdida.** | Un ROE positivo indica generación de rentabilidad sobre el patrimonio. | **Mayor es mejor** |
| **ROA (Resultados del ejercicio / Activo promedio)** | Mide la rentabilidad obtenida sobre el activo promedio. | Sin máximo regulatorio general identificado. **0% = ausencia de rentabilidad; <0% = pérdida.** | Un ROA positivo indica capacidad de generar resultados a partir de los activos. | **Mayor es mejor** |
| **Cartera bruta / (Depósitos a la vista + Depósitos a plazo)** | Mide la relación entre la cartera de crédito y los depósitos considerados como fuente de fondeo. | **100% = cartera equivalente a los depósitos considerados**. No constituye un máximo regulatorio general. | Valores elevados indican mayor intermediación crediticia, aunque niveles excesivos deben analizarse conjuntamente con la liquidez. | **Mayor es generalmente mejor** |
| **Fondos disponibles / Total depósitos a corto plazo** | Mide la capacidad de cubrir los depósitos de corto plazo mediante fondos disponibles. | **100% = fondos disponibles equivalentes a los depósitos de corto plazo**. No constituye un máximo regulatorio general. | Un mayor porcentaje representa un mayor colchón de liquidez inmediata, aunque niveles excesivamente altos pueden reflejar recursos poco utilizados. | **Mayor generalmente es mejor** |

### Indicadores donde valores más altos representan un mejor desempeño

 Se agrupan para el análisis los indicadores para los cuales, desde una perspectiva
 financiera, valores mayores representan generalmente un mejor desempeño.

In [ ]:
indicadores_mayor=['SUFICIENCIA PATRIMONIAL',
                   'INDICE DE CAPITALIZACION NETO: FK / FI',
                    'ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS',
                    'ACTIVOS PRODUCTIVOS / PASIVOS CON COSTO',
                    'COBERTURA DE LA CARTERA REFINANCIADA',
                    'COBERTURA DE LA CARTERA REESTRUCTURADA',
                    'COBERTURA DE LA CARTERA PROBLEMÁTICA',
                    'RESULTADOS DEL EJERCICIO / PATRIMONIO PROMEDIO',
                    'RESULTADOS DEL EJERCICIO / ACTIVO PROMEDIO',
                    'CARTERA BRUTA / (DEPOSITOS A LA VISTA + DEPOSITOS A PLAZO)',
                    'FONDOS DISPONIBLES / TOTAL DEPOSITOS A CORTO PLAZO']

In [ ]:
indicadores_notebook = [
    # Solvencia
    'SUFICIENCIA PATRIMONIAL',
    # Calidad de activos
    'INDICE DE CAPITALIZACION NETO: FK / FI',
    # Cobertura
    'COBERTURA DE LA CARTERA PROBLEMÁTICA',
    # Rentabilidad
    'RESULTADOS DEL EJERCICIO / PATRIMONIO PROMEDIO',
    # Liquidez
    'FONDOS DISPONIBLES / TOTAL DEPOSITOS A CORTO PLAZO'
]

In [ ]:
# Diccionario de meses
meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

# Crear número de mes
base_final["MES_NUM"] = base_final["MES"].str.lower().map(meses)

# Crear fecha
base_final["FECHA"] = pd.to_datetime(
    base_final["AÑO"].astype(str) + "-" +
    base_final["MES_NUM"].astype(str) + "-01"
)

#### Series temporales 

In [ ]:
eda.graficar_indicadores(
    base_final,
    indicadores_mayor,
     carpeta=RAIZ_PROYECTO / "images",
    indicadores_notebook=indicadores_notebook
)


Durante el análisis exploratorio se identificaron valores atípicos particularmente elevados en los indicadores de suficiencia patrimonial y cobertura de la cartera problemática. Sin embargo, estos valores corresponden a Citibank, N.A. Sucursal Ecuador, una entidad con características estructurales diferentes a las del resto de bancos analizados, debido a su condición de banco privado extranjero y a su enfoque principalmente corporativo y empresarial, en lugar de concentrarse en el segmento minorista, como el resto de las entidades. 

Por tanto, estos valores extremos no se consideran inicialmente errores de registro ni se eliminan de la base, ya que pueden reflejar las particularidades de su estructura financiera y de los denominadores utilizados en el cálculo de estos indicadores. En consecuencia, los valores serán conservados para las etapas posteriores de análisis, considerando su posible influencia sobre la distribución de las variables y sobre las técnicas de reducción de dimensionalidad y clustering. 

Con esta consideración, se continúa con el análisis mediante boxplots y mapas de calor (heatmaps) para evaluar la distribución, dispersión y relaciones entre los indicadores.

#### Heatmaps

In [ ]:
eda.generar_heatmaps(base_final, indicadores_mayor, indicadores_notebook=indicadores_notebook, carpeta=RAIZ_PROYECTO / "images")

El análisis mediante mapas de calor permitió identificar diferencias importantes en el comportamiento temporal de los indicadores financieros entre las entidades bancarias. 

Para el indicador de  suficiencia patrimonial se observa una marcada heterogeneidad entre instituciones, con entidades que mantienen niveles elevados durante gran parte del período y otras que presentan valores inferiores al umbral regulatorio del 9%, evidenciando además cambios en determinados períodos. En el índice de capitalización neto se aprecia una mayor estabilidad relativa, aunque persisten diferencias estructurales entre bancos, destacándose algunas entidades con niveles consistentemente superiores al resto. 

Por su parte, la cobertura de la cartera problemática presenta una elevada dispersión entre instituciones, con valores generalmente inferiores a 100% en varias entidades y niveles considerablemente superiores en otras, particularmente en Citibank que es el mayor de todos los atípicos. 

Finalmente, el indicador de fondos disponibles respecto al total de depósitos de corto plazo muestra variaciones tanto entre entidades, permitiendo identificar diferentes patrones de liquidez. 

En conjunto, los heatmaps evidencian que las entidades bancarias no presentan un comportamiento financiero homogéneo, sino que existen diferencias persistentes y patrones temporales específicos, lo que constituye un primer indicio de la existencia de perfiles financieros diferenciados que serán posteriormente analizados mediante técnicas de reducción de dimensionalidad y clustering.

#### Diagramas de Caja (Boxplots)

In [ ]:
eda.generar_boxplots(base_final, indicadores_mayor, indicadores_notebook=indicadores_notebook, carpeta=RAIZ_PROYECTO / "images")

Los diagramas de caja evidencian una importante heterogeneidad en la distribución de los indicadores financieros analizados, así como la presencia de valores atípicos en varias de las variables. 

* La suficiencia patrimonial presenta una distribución altamente asimétrica, con numerosos valores extremos y observaciones considerablemente alejadas del conjunto principal, destacándose nuevamente los valores asociados a Citibank. 
* El índice de capitalización neto muestra una dispersión más acotada, aunque presenta múltiples observaciones por encima del límite superior del diagrama, lo que indica la existencia de entidades con niveles de capitalización significativamente superiores al comportamiento central de la muestra. 
* En la cobertura de la cartera problemática se observa una marcada concentración de las observaciones en niveles relativamente bajos y una gran cantidad de valores extremos positivos, asociados principalmente a diferencias importantes en la estructura de las entidades y, particularmente, al comportamiento de Citibank. 
* Por su parte, el indicador de resultados del ejercicio respecto al patrimonio promedio presenta una distribución más concentrada alrededor de valores positivos, aunque se identifican numerosos valores atípicos negativos que pueden corresponder a períodos con resultados desfavorables. 
* Finalmente, el indicador de fondos disponibles respecto al total de depósitos de corto plazo presenta una dispersión moderada y una concentración de observaciones alrededor de su distribución central, acompañada de varios valores extremos superiores. 

En conjunto, los diagramas de caja confirman la existencia de distribuciones heterogéneas, asimetrías y observaciones extremas, por lo que estos valores deberán ser analizados individualmente para determinar si corresponden a errores de datos, eventos particulares o características financieras propias de las entidades, antes de definir cualquier tratamiento para las etapas posteriores del análisis multivariado.

#### Identificación de atípicos por el método del rango intercuartílico

In [ ]:
tabla_atipicos = eda.detectar_atipicos_iqr(
    base_final,
    indicadores_mayor
)

In [ ]:
conteo_atipicos = pd.pivot_table(
    tabla_atipicos,
    index=["ENTIDAD", "INDICADOR"],
    columns="TIPO_ATIPICO",
    values="FECHA",
    aggfunc="count",
    fill_value=0
).reset_index()

# Total de atípicos
conteo_atipicos["TOTAL_ATIPICOS"] = (
    conteo_atipicos.get("Inferior", 0)
    + conteo_atipicos.get("Superior", 0)
)

# Ordenar de mayor a menor
conteo_atipicos = conteo_atipicos.sort_values(
    "TOTAL_ATIPICOS",
    ascending=False
).reset_index(drop=True)


In [ ]:
# Número de valores no nulos por entidad e indicador
total_valores = (
    base_final
    .groupby("ENTIDAD")[indicadores_mayor]
    .count()
    .reset_index()
    .melt(
        id_vars="ENTIDAD",
        var_name="INDICADOR",
        value_name="TOTAL_VALORES"
    )
)

# Unir con tabla de atípicos
atipicos_f = conteo_atipicos.merge(
    total_valores,
    on=["ENTIDAD", "INDICADOR"],
    how="left"
)

# Porcentaje de valores atípicos problemáticos
atipicos_f["PORC_ATIPICOS_INF"] = (
    atipicos_f["Inferior"]
    / atipicos_f["TOTAL_VALORES"]
    * 100
)

In [ ]:
atipicos_f

In [ ]:
atipicos_f = atipicos_f.sort_values(
    "PORC_ATIPICOS_INF",
    ascending=False
)

In [ ]:
atipicos_f

La identificación de valores atípicos mediante el criterio del rango intercuartílico no se utilizó como criterio automático para excluir entidades de la muestra. Debido a que determinados valores extremos pueden responder a características estructurales propias de las entidades y no necesariamente a errores de registro, las observaciones identificadas fueron sometidas a un proceso de validación. 

 En algunos casos, una misma entidad presenta valores atípicos durante una proporción elevada del período analizado, lo que sugiere la existencia de diferencias estructurales en su comportamiento financiero y no necesariamente errores en los datos. Un caso notable es BP CITIBANK, que para el indicador de Suficiencia Patrimonial, registra 44 observaciones atípicas inferiores de un total de 51, es decir por debajo del $Q1-1,5\times iqr$ o el BP FINCA S.A./BANCO AMIBANK S.A. que fue liquidado por mal manejo de sus fondos . Estos comportamientos evidencian un perfil financiero diferenciado respecto al resto de entidades y demuestra que los valores atípicos pueden representar características estructurales propias de una institución.

Asimismo, se observa que la dirección de los valores atípicos depende del indicador: mientras algunas variables presentan principalmente observaciones atípicas inferiores, otras concentran sus valores extremos en la parte superior de la distribución, siendo indicadores que entre más altos sean los valores implican un mejor desempeño. 
Por tanto, la identificación de atípicos se utilizará como mecanismo de diagnóstico y no como criterio automático de eliminación de observaciones o variables. Dado que el objetivo del estudio es identificar perfiles financieros mediante técnicas de reducción de dimensionalidad y clustering, no se eliminará ningún indicador por presentar una elevada proporción de valores atípicos, ya que estos pueden contener información relevante para diferenciar las entidades bancarias. En consecuencia, se conservará la información observada y, debido a la importancia de los valores extremos en las técnicas posteriores, los datos serán centralizados antes de aplicar la reducción de dimensionalidad y el clustering, buscando disminuir la influencia de las diferencias de escala y mantener la información contenida en las observaciones extremas.

In [ ]:
# --------------------------------------------------------
# Registro de decisiones del EDA
# --------------------------------------------------------

log_eda = {
    "proceso": "Análisis Exploratorio de Datos",
    "fecha_ejecucion": datetime.now().isoformat(),

    "secciones": {

        "valores_altos_mejor_desempeno": {
            "filas_eliminadas": 0,
            "variables_eliminadas": [],
            "metodo": "Rango intercuartílico (IQR)",
            "decision": "Los valores atípicos fueron conservados."
        }
    },

    "base_final": {
        "filas": int(base_final.shape[0]),
        "columnas": int(base_final.shape[1])
    }
}

### Indicadores donde valores más bajos representan un mejor desempeño

 Se agrupan para el análisis los indicadores para los cuales, desde una perspectiva
 financiera, valores menores representan generalmente un mejor desempeño.

In [ ]:
base_final.columns.to_list()

In [ ]:
indicadores_menor=['ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS',
                   'MOROSIDAD DE LA CARTERA TOTAL',
                    'MOROSIDAD DE LA CARTERA DE CREDITOS CONSUMO',
                    'MOROSIDAD DE LA CARTERA DE CRÉDITOS INMOBILIARIO',
                    'MOROSIDAD DE LA CARTERA DE CRÉDITOS MICROCRÉDITO',
                    'GASTOS DE OPERACION ESTIMADOS / TOTAL ACTIVO PROMEDIO (3)',
                    'GASTOS DE OPERACION  / MARGEN FINANCIERO',
                    'GASTOS DE PERSONAL ESTIMADOS / ACTIVO PROMEDIO (3)']

In [ ]:
indicadores_notebook = [
    'ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS',
    'MOROSIDAD DE LA CARTERA TOTAL',
    'GASTOS DE OPERACION  / MARGEN FINANCIERO'
]

In [ ]:
# Diccionario de meses
meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

# Crear número de mes
base_final["MES_NUM"] = base_final["MES"].str.lower().map(meses)

# Crear fecha
base_final["FECHA"] = pd.to_datetime(
    base_final["AÑO"].astype(str) + "-" +
    base_final["MES_NUM"].astype(str) + "-01"
)

#### Series temporales

In [ ]:
eda.graficar_indicadores(
    base_final,
    indicadores_menor,
    carpeta=RAIZ_PROYECTO / "images",
    indicadores_notebook=indicadores_notebook
)


El análisis de la evolución temporal evidencia diferencias importantes entre las entidades bancarias en indicadores asociados con distintos componentes de su perfil financiero. En particular, la morosidad de la cartera total muestra una marcada heterogeneidad: mientras varias entidades mantienen niveles relativamente bajos y estables, algunas presentan niveles considerablemente superiores. Destaca BP FINCA S.A./Banco Amibank S.A., cuya morosidad supera el 30% durante varios períodos, seguida por BP D-MIRO S.A./Banco Atlántida S.A., que alcanza niveles superiores al 20% en determinados períodos.

Esta diferenciación también se observa en la proporción de activos improductivos netos sobre el total de activos. Entidades como BP BANCO COMERCIAL DE MANABÍ y BP CAPITAL presentan niveles elevados y trayectorias diferenciadas respecto a otras instituciones, que se mantienen en rangos considerablemente menores. Esto evidencia que las entidades presentan estructuras distintas en cuanto a la composición y calidad de sus activos.

Por otra parte, el indicador de gastos de operación respecto al margen financiero presenta una dispersión particularmente elevada, con episodios extremos en determinadas entidades, lo que evidencia diferencias importantes en la relación entre los gastos operativos y los ingresos generados por la actividad financiera. Estos valores extremos deberán ser considerados posteriormente en el análisis de los datos, sin asumir inicialmente que corresponden a errores, debido a la naturaleza del indicador y a la posible influencia de variaciones en el margen financiero.

En conjunto, las tres variables evidencian que las entidades bancarias presentan comportamientos diferenciados no solo en términos de morosidad, sino también respecto a la calidad de sus activos y su eficiencia operativa. Por ejemplo, mientras algunas entidades presentan niveles elevados de morosidad, otras destacan por una mayor proporción de activos improductivos o por una mayor variabilidad en sus gastos de operación. Estas diferencias sugieren la existencia de distintos perfiles financieros entre las instituciones analizadas y constituyen evidencia exploratoria que respalda la aplicación posterior de técnicas de reducción de dimensionalidad y clustering para identificar y caracterizar dichos perfiles.

#### Heatmaps

In [ ]:
eda.generar_heatmaps(base_final, indicadores_menor, indicadores_notebook=indicadores_notebook, carpeta=RAIZ_PROYECTO / "images")

Los tres heatmaps evidencian una marcada heterogeneidad entre las entidades bancarias privadas a lo largo del período analizado, tanto en términos de calidad de activos, riesgo de crédito como de eficiencia operativa. En cuanto a los activos improductivos netos sobre el total de activos, se observan entidades con niveles persistentemente elevados, como BP BANCO COMERCIAL DE MANABÍ y BP CAPITAL, cuyos valores alcanzan niveles superiores al 30% durante determinados períodos, mientras que otras entidades, como BP COOPNACIONAL, BP GENERAL RUMIÑAHUI y BP VISIONFUND ECUADOR S.A., mantienen proporciones considerablemente menores.

Las diferencias son aún más marcadas en la morosidad de la cartera total. BP FINCA S.A./BANCO AMIBANK S.A. presenta los niveles más elevados, superando el 30% durante varios períodos, mientras que BP D-MIRO S.A./BANCO ATLÁNTIDA S.A. alcanza niveles superiores al 20% en determinados momentos. En contraste, entidades como BP AMAZONAS, BP AUSTRO, BP GUAYAQUIL y BP PICHINCHA mantienen niveles considerablemente menores durante gran parte del período. Esto evidencia que las entidades presentan perfiles diferenciados respecto al riesgo de crédito y la calidad de su cartera.

Finalmente, el indicador de gastos de operación sobre margen financiero presenta una distribución considerablemente más dispersa, con episodios extremos concentrados principalmente en BP D-MIRO S.A./BANCO ATLÁNTIDA S.A. y otros períodos puntuales en distintas entidades. Esta elevada variabilidad muestra diferencias en la relación entre los gastos operativos y el margen generado por la actividad financiera y, al mismo tiempo, evidencia la necesidad de considerar cuidadosamente los valores extremos antes de aplicar las técnicas de reducción de dimensionalidad.

En conjunto, los tres indicadores muestran que las entidades no presentan un comportamiento financiero homogéneo: algunas se caracterizan por mayores niveles de activos improductivos, otras por una mayor exposición al riesgo de crédito y otras por comportamientos diferenciados en términos de eficiencia operativa. Por ejemplo, BP FINCA S.A./BANCO AMIBANK S.A. destaca por sus elevados niveles de morosidad y activos improductivos, mientras que BP CAPITAL y BP BANCO COMERCIAL DE MANABÍ presentan niveles elevados de activos improductivos. Estas diferencias sugieren la existencia de estructuras y comportamientos financieros diferenciados entre las instituciones, lo que constituye evidencia exploratoria relevante para la identificación de perfiles financieros mediante técnicas de reducción de dimensionalidad y clustering.

#### Diagramas de Caja (Boxplots)

In [ ]:
eda.generar_boxplots(base_final, indicadores_menor, indicadores_notebook=indicadores_notebook, carpeta=RAIZ_PROYECTO / "images")

Los diagramas de caja evidencian una importante presencia de valores atípicos superiores al límite de $Q3 + 1.5 \times IQR$, especialmente en los indicadores donde un menor valor representa un mejor desempeño. En activos improductivos netos / total activos, se observa una distribución con numerosos valores por encima del límite superior, alcanzando niveles superiores al 40%, lo que evidencia la existencia de entidades con proporciones de activos improductivos significativamente mayores al comportamiento central de la muestra.

En morosidad de la cartera total, la concentración de valores atípicos superiores es aún más evidente. La distribución central se mantiene en niveles relativamente bajos, pero existe un número considerable de observaciones que superan el límite superior, llegando a valores superiores al 30%. Estos resultados muestran que determinadas entidades presentan niveles de morosidad considerablemente superiores al comportamiento general.

Finalmente, gastos de operación / margen financiero presenta la mayor dispersión y una cantidad importante de valores extremos tanto positivos como negativos. Destacan observaciones que superan ampliamente el límite superior, con valores superiores al 10.000%, lo que evidencia una distribución altamente asimétrica y requiere especial atención antes de las técnicas posteriores.

En conjunto, los resultados muestran que los valores atípicos superiores son relevantes en los indicadores asociados con una evaluación desfavorable hacia valores altos. Estos valores no serán eliminados automáticamente, ya que pueden representar diferencias estructurales entre las entidades y, por tanto, aportar información relevante para la identificación de perfiles financieros mediante reducción de dimensionalidad y clustering.

#### Identificación de atípicos por el método de rango intercuartílico

In [ ]:
tabla_atipicos = eda.detectar_atipicos_iqr(
    base_final,
    indicadores_menor
)

In [ ]:
conteo_atipicos = pd.pivot_table(
    tabla_atipicos,
    index=["ENTIDAD", "INDICADOR"],
    columns="TIPO_ATIPICO",
    values="FECHA",
    aggfunc="count",
    fill_value=0
).reset_index()

# Total de atípicos
conteo_atipicos["TOTAL_ATIPICOS"] = (
    conteo_atipicos.get("Inferior", 0)
    + conteo_atipicos.get("Superior", 0)
)

# Ordenar de mayor a menor
conteo_atipicos = conteo_atipicos.sort_values(
    "TOTAL_ATIPICOS",
    ascending=False
).reset_index(drop=True)


In [ ]:
# Número de valores no nulos por entidad e indicador
total_valores = (
    base_final
    .groupby("ENTIDAD")[indicadores_mayor]
    .count()
    .reset_index()
    .melt(
        id_vars="ENTIDAD",
        var_name="INDICADOR",
        value_name="TOTAL_VALORES"
    )
)

# Unir con tabla de atípicos
atipicos_f = conteo_atipicos.merge(
    total_valores,
    on=["ENTIDAD", "INDICADOR"],
    how="left"
)

# Porcentaje de valores atípicos problemáticos
atipicos_f["PORC_ATIPICOS_SUP"] = (
    atipicos_f["Superior"]
    / atipicos_f["TOTAL_VALORES"]
    * 100
)

In [ ]:
atipicos_f

In [ ]:
atipicos_f = atipicos_f.sort_values(
    "PORC_ATIPICOS_SUP",
    ascending=False
)

In [ ]:
atipicos_f

Dado que la variable de morosidad de la cartera total constituye una medida agregada del nivel de morosidad de cada entidad, se utilizará este indicador como referencia para caracterizar el riesgo de mora. 

En consecuencia, se excluirán del conjunto de variables  los indicadores específicos de morosidad específicos de cada cartera, como el de de consumo, el inmobiliario y el microcrédito. 

Esta decisión responde a las diferencias existentes en la estructura y especialización de las entidades bancarias, dado que determinados segmentos de cartera no tienen la misma relevancia o incluso pueden no formar parte de la actividad de algunas instituciones. Por ejemplo, el caso de BP Solidario presenta atípicos en morosidad inmobiliaria, siendo esta una cartera para la entidad con 0 operaciones en el mercado, puesto que los productos que ofrece son más inclinado al microcrédito y a al crédito de consumo, razón por la cual  puede generar valores que no representan adecuadamente el comportamiento de la entidad en dicho indicador. 

Por esta razón, se considera más consistente utilizar la morosidad de la cartera total, que permite evaluar de manera homogénea el nivel de morosidad de las entidades independientemente de la composición particular de sus carteras.

In [ ]:
base_sin_eliminar_col=base_final.copy()

In [ ]:
columnas_eliminar = [
    'MOROSIDAD DE LA CARTERA DE CREDITOS CONSUMO',
    'MOROSIDAD DE LA CARTERA DE CRÉDITOS INMOBILIARIO',
    'MOROSIDAD DE LA CARTERA DE CRÉDITOS MICROCRÉDITO'
]

base_final = base_final.drop(columns=columnas_eliminar)

In [ ]:
base_final.shape

In [ ]:
base_final

In [ ]:
log_eda["secciones"]["valores_bajos_mejor_desempeno"] = {
    "filas_eliminadas": 0,

    "variables_eliminadas": columnas_eliminar,

    "cantidad_variables_eliminadas": len(columnas_eliminar),

    "metodo": "Análisis exploratorio y evaluación de comparabilidad",

    "decision": (
        "Se eliminaron los indicadores de morosidad por segmento "
        "debido a diferencias en la especialización de las entidades "
        "y a la falta de comparabilidad entre instituciones."
    )
}

In [ ]:
log_eda

In [ ]:
log_eda["base_final"]={'filas': base_final.shape[0], 'columnas': base_final.shape[1]}

In [ ]:
log_eda

## Indicadores de porcentaje de cartera

Los indicadores de porcentaje de cartera permiten analizar la composición y calidad de la cartera de crédito de las entidades financieras. Estas métricas expresan la participación de diferentes componentes de la cartera respecto de la cartera bruta, permitiendo identificar la proporción de operaciones que presentan características asociadas al deterioro, vencimiento, refinanciamiento o reestructuración.

Para este análisis se consideran los siguientes indicadores: cartera improductiva, cartera vencida, cartera que no devenga intereses, cartera refinanciada y cartera reestructurada, todos expresados como proporción de la cartera bruta. Su análisis permite comparar la estructura y calidad de la cartera entre entidades y a través del tiempo.

| Variable | Significado | Límite / referencia | Interpretación | Dirección deseable |
| --- | --- | --- | --- | --- |
| **Cartera improductiva / Cartera Bruta** | Mide la proporción de la cartera improductiva respecto a la cartera bruta total. | No existe un límite regulatorio general específico. | Valores elevados indican una mayor participación de cartera que no genera ingresos financieros o presenta deterioro. | **Menor es mejor** |
| **Cartera vencida / Cartera Bruta** | Mide la proporción de la cartera vencida respecto a la cartera bruta total. | No existe un límite regulatorio general específico. | Valores elevados indican una mayor proporción de créditos que presentan incumplimiento de pago. | **Menor es mejor** |
| **Cartera que no devenga intereses / Cartera bruta** | Mide la proporción de la cartera que no genera intereses respecto a la cartera bruta. | No existe un límite regulatorio general específico. | Valores elevados indican una mayor proporción de cartera que ha dejado de generar ingresos por intereses. | **Menor es mejor** |
| **Cartera refinanciada / Cartera bruta** | Mide la participación de la cartera refinanciada dentro de la cartera bruta. | No existe un límite regulatorio general específico. | Valores elevados indican una mayor participación de créditos cuyas condiciones originales han sido modificadas mediante refinanciamiento. | **Menor es mejor** |
| **Cartera reestructurada / Cartera bruta** | Mide la participación de la cartera reestructurada dentro de la cartera bruta. | No existe un límite regulatorio general específico. | Valores elevados indican una mayor participación de créditos cuyas condiciones han sido modificadas mediante procesos de reestructuración. | **Menor es mejor** |

In [ ]:
indicadores_cartera=['CARTERA IMPRODUCTIVA / CARTERA BRUTA',	
                   'CARTERA VENCIDA / CARTERA BRUTA',	
                   'CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA',	
                   'CARTERA REFINANCIADA / CARTERA BRUTA',	
                   'CARTERA REESTRUCTURADA / CARTERA BRUTA']

In [ ]:
indicadores_notebook = indicadores_cartera

#### Series temporales

In [ ]:
eda.graficar_indicadores(
    base_final,
    indicadores_cartera,
    carpeta=RAIZ_PROYECTO / "images",
    indicadores_notebook=indicadores_notebook
)


Las series temporales muestran que, aunque la dirección deseable de estos indicadores corresponde a valores bajos, existen diferencias importantes entre las entidades. Algunas instituciones presentan niveles considerablemente superiores al resto, particularmente en los indicadores de cartera improductiva, cartera vencida y cartera que no devenga intereses. Sin embargo, los valores observados se mantienen por debajo del 100%, por lo que no se evidencian valores extremos en términos del rango porcentual de la razón. La principal característica identificada es la heterogeneidad entre entidades y la presencia de episodios de incremento en determinados períodos, lo que refleja diferentes niveles y dinámicas de riesgo y calidad de cartera.

Respecto a la cartera improductiva/cartera bruta (que no genera ingresos) destaca BP FINCA S.A./BANCO AMIBANK S.A., con valores superiores al 30%, mientras que otras entidades mantienen niveles considerablemente menores. En cartera vencida/cartera bruta también existe heterogeneidad. BP FINCA S.A./BANCO AMIBANK S.A. presenta una trayectoria creciente que alcanza aproximadamente 15%, mientras BP D-MIRO S.A./BANCO ATLÁNTIDA S.A. registra episodios cercanos al 10%.
En Cartera reestructurada / Cartera Bruta, BP FINCA S.A./BANCO AMIBANK S.A. presenta los niveles más elevados, alcanzando aproximadamente 11–12%, mientras que BP DINERS muestra una trayectoria creciente que llega a cerca del 7–8%.

#### Heatmaps

In [ ]:
eda.generar_heatmaps(base_final, indicadores_cartera, indicadores_notebook=indicadores_notebook, carpeta=RAIZ_PROYECTO / "images")

Los heatmaps evidencian una marcada heterogeneidad entre las entidades bancarias y a lo largo del período analizado. En general, predominan valores relativamente bajos; sin embargo, se identifican entidades que mantienen niveles elevados o presentan incrementos importantes durante determinados períodos. 
En particular, BP FINCA S.A./BANCO AMIBANK S.A. presenta niveles persistentemente altos de cartera improductiva y cartera que no devenga intereses, alcanzando aproximadamente 30% y 20%, respectivamente, durante varios períodos. De igual manera, BP D-MIRO S.A./BANCO ATLÁNTIDA S.A. registra valores elevados y sostenidos en estos indicadores, especialmente entre 2023 y 2025. 
Para la cartera vencida, nuevamente destacan estas dos entidades, con BP FINCA S.A./BANCO AMIBANK S.A. alcanzando valores cercanos al 15%. En cuanto a la cartera refinanciada, se observan niveles particularmente elevados en BP BANCO DESARROLLO DE LOS PUEBLOS S.A., CODESARROLLO, BP D-MIRO S.A./BANCO ATLÁNTIDA S.A., BP DELBANK y BP LITORAL. 

Finalmente, la cartera reestructurada presenta mayores niveles en BP FINCA S.A./BANCO AMIBANK S.A. y BP DINERS. En conjunto, los patrones observados muestran que el deterioro y la composición de la cartera no se distribuyen de manera homogénea entre las entidades, sino que existen perfiles diferenciados y persistencia temporal en determinados casos.

#### Diagramas de caja (Boxplots)

In [ ]:
eda.generar_boxplots(base_final, indicadores_cartera, indicadores_notebook=indicadores_notebook, carpeta=RAIZ_PROYECTO / "images")

Los diagramas de caja evidencian la presencia de numerosos valores atípicos por encima del límite superior definido mediante el criterio de $Q3 + 1.5 \times IQR$ en los cinco indicadores analizados. Esta concentración de observaciones por encima del bigote superior confirma una distribución con asimetría hacia valores elevados, particularmente en indicadores cuya dirección deseable corresponde a valores bajos. La mayor dispersión se observa en la cartera improductiva, donde se identifican valores atípicos que superan el 30%, seguida de la cartera que no devenga intereses, con valores superiores al 20%. Asimismo, la cartera refinanciada presenta valores atípicos cercanos al 19%, mientras que la cartera vencida alcanza aproximadamente el 15% y la cartera reestructurada supera el 11%. Estos resultados evidencian la existencia de observaciones extremas que corresponden a períodos y entidades con niveles relativamente elevados respecto al comportamiento general de la muestra.

#### Identificación de atípicos por el método del rango intercuartílico

In [ ]:
tabla_atipicos = eda.detectar_atipicos_iqr(
    base_final,
    indicadores_cartera
)

In [ ]:
conteo_atipicos = pd.pivot_table(
    tabla_atipicos,
    index=["ENTIDAD", "INDICADOR"],
    columns="TIPO_ATIPICO",
    values="FECHA",
    aggfunc="count",
    fill_value=0
).reset_index()

# Total de atípicos
conteo_atipicos["TOTAL_ATIPICOS"] = (
    conteo_atipicos.get("Inferior", 0)
    + conteo_atipicos.get("Superior", 0)
)

# Ordenar de mayor a menor
conteo_atipicos = conteo_atipicos.sort_values(
    "TOTAL_ATIPICOS",
    ascending=False
).reset_index(drop=True)


In [ ]:
# Número de valores no nulos por entidad e indicador
total_valores = (
    base_final
    .groupby("ENTIDAD")[indicadores_cartera]
    .count()
    .reset_index()
    .melt(
        id_vars="ENTIDAD",
        var_name="INDICADOR",
        value_name="TOTAL_VALORES"
    )
)

# Unir con tabla de atípicos
atipicos_f = conteo_atipicos.merge(
    total_valores,
    on=["ENTIDAD", "INDICADOR"],
    how="left"
)

# Porcentaje de valores atípicos problemáticos
atipicos_f["PORC_ATIPICOS_SUP"] = (
    atipicos_f["Superior"]
    / atipicos_f["TOTAL_VALORES"]
    * 100
)

In [ ]:
atipicos_f

In [ ]:
atipicos_f = atipicos_f.sort_values(
    "PORC_ATIPICOS_SUP",
    ascending=False
)

In [ ]:
atipicos_f

El análisis mediante el criterio de $Q3+1,5\times IQR$ evidencia una concentración de valores atípicos superiores en determinadas entidades, particularmente en Banco Amibank, Banco D-MIRO S.A./Banco Atlántida y Banco Diners Club. En el caso de Amibank, la elevada frecuencia de atípicos en cartera improductiva, cartera que no devenga intereses y cartera vencida es consistente con los elevados niveles de morosidad registrados por la entidad durante el período analizado. D-MIRO también presenta una concentración importante de observaciones atípicas asociadas con la calidad de su cartera, coherente con los niveles históricamente elevados de morosidad reportados por la institución. Por su parte, Diners destaca específicamente por la proporción de observaciones atípicas en cartera reestructurada, con 27 de 51 observaciones (52,94%). 

Este resultado debe interpretarse como una característica estadística diferenciada de la entidad y no necesariamente como un atípico por error de registro a eliminar, considerando su elevada especialización en cartera de consumo y la presencia de montos significativos de cartera reestructurada. En consecuencia, los valores atípicos identificados se conservarán para las etapas posteriores del análisis, dado que pueden representar características estructurales de las entidades y contribuir a la identificación de perfiles financieros diferenciados mediante las técnicas multivariadas.

## Indicadores de crecimiento 


Los indicadores de crecimiento permiten evaluar la evolución de las principales cuentas financieras de las entidades a través del tiempo. Estas métricas muestran la variación porcentual de determinados componentes del activo, las fuentes de fondeo y el patrimonio, permitiendo identificar cambios en el tamaño y en la estructura financiera de las instituciones.

En términos generales, un valor positivo indica un crecimiento de la cuenta respecto al período anterior, mientras que un valor negativo representa una disminución. Por tanto, para las variables analizadas, una evolución positiva puede interpretarse como un incremento en los recursos o componentes financieros correspondientes, aunque su conveniencia debe analizarse de acuerdo con la naturaleza de cada cuenta y en conjunto con los demás indicadores financieros.

En este análisis se consideran las variaciones de fondos disponibles, inversiones, cartera de créditos, total de activos, depósitos a la vista, depósitos a plazo y patrimonio. El análisis conjunto de estas variables permite identificar diferencias en la dinámica de crecimiento de las entidades y detectar períodos de expansión o contracción.

| Variable | Significado | Límite / referencia | Interpretación | Dirección deseable |
| --- | --- | --- | --- | --- |
| **Crecimiento de activos totales** | Mide la variación porcentual interanual de los activos totales de la entidad. | No existe un límite regulatorio general. El **0%** representa ausencia de crecimiento. | Un valor **positivo** indica expansión de los activos; un valor **negativo** indica una contracción. Un crecimiento elevado puede reflejar expansión, pero también debe analizarse junto con la calidad y rentabilidad de los activos. | **Positivo y sostenible** |
| **Crecimiento de cartera de créditos** | Mide la variación porcentual interanual de la cartera de créditos. | No existe un límite regulatorio general. El **0%** representa estabilidad respecto al año anterior. | Un valor **positivo** indica expansión de la colocación de crédito; un valor **negativo** indica reducción de la cartera. Un crecimiento elevado puede representar mayor intermediación, pero debe analizarse conjuntamente con la morosidad y calidad de cartera. | **Positivo y sostenible** |
| **Crecimiento de depósitos a la vista** | Mide la variación porcentual interanual de los depósitos a la vista. | No existe un límite regulatorio general. El **0%** representa estabilidad. | Un valor **positivo** indica un incremento de los recursos captados a la vista; un valor **negativo** indica una disminución. Un crecimiento positivo fortalece la disponibilidad de fuentes de fondeo, aunque debe evaluarse su estabilidad. | **Positivo y sostenible** |
| **Crecimiento de depósitos a plazo** | Mide la variación porcentual interanual de los depósitos a plazo. | No existe un límite regulatorio general. El **0%** representa estabilidad. | Un valor **positivo** indica un aumento de las captaciones a plazo; un valor **negativo** indica una reducción. Un crecimiento positivo puede contribuir a fortalecer el fondeo, aunque implica considerar el costo financiero asociado. | **Positivo y sostenible** |
| **Crecimiento de patrimonio** | Mide la variación porcentual interanual del patrimonio de la entidad. | No existe un límite regulatorio general de crecimiento. El **0%** representa estabilidad. | Un valor **positivo** indica fortalecimiento patrimonial; un valor **negativo** indica una reducción del patrimonio. El crecimiento patrimonial sostenido contribuye a fortalecer la capacidad de absorción de pérdidas y respaldar la expansión de activos. | **Positivo y sostenible** |
| **Crecimiento de fondos disponibles** | Mide la variación porcentual interanual de los fondos disponibles. | No existe un límite regulatorio general de crecimiento. El **0%** representa estabilidad. | Un valor **positivo** indica un aumento de los recursos líquidos disponibles; un valor **negativo** indica una disminución. Un crecimiento positivo puede mejorar la posición de liquidez, aunque niveles excesivos podrían reflejar recursos con menor utilización productiva. | **Positivo, pero equilibrado** |
| **Crecimiento de inversiones** | Mide la variación porcentual interanual de las inversiones financieras de la entidad. | No existe un límite regulatorio general de crecimiento. El **0%** representa estabilidad. | Un valor **positivo** indica expansión de las inversiones; un valor **negativo** indica reducción. El crecimiento debe analizarse considerando su composición, rentabilidad, liquidez y riesgo. | **Positivo y sostenible** |

In [ ]:
indicadores_balance=['VAR_FONDOS DISPONIBLES',	
                     'VAR_INVERSIONES',	'VAR_CARTERA DE CRÉDITOS',	
                     'VAR_TOTAL ACTIVO',	'VAR_Depósitos a la vista',
                    'VAR_Depósitos a plazo',	
                    'VAR_TOTAL PATRIMONIO']


In [ ]:
indicadores_notebook = [
    'VAR_TOTAL ACTIVO',
    'VAR_CARTERA DE CRÉDITOS',
    'VAR_TOTAL PATRIMONIO'
]

#### Series temporales

In [ ]:
eda.graficar_indicadores(
    base_final,
    indicadores_balance,
    carpeta=RAIZ_PROYECTO / "images",
    indicadores_notebook=indicadores_notebook
)


Las variables de crecimiento interanual evidencian una marcada heterogeneidad en la dinámica financiera de las entidades. Se identifican instituciones con trayectorias relativamente estables y variaciones moderadas, como BP Guayaquil, BP Pichincha, BP Produbanco, BP Internacional, BP Loja, BP Machala y BP General Rumiñahui, cuyos crecimientos de activos, cartera y patrimonio se mantienen generalmente dentro de rangos relativamente acotados durante el período analizado. En contraste, se observan entidades con mayor variabilidad, particularmente BP Capital, BP D-MIRO S.A./Banco Atlántida S.A., BP FINCA S.A./Banco Amibank S.A. y BP Banco Comercial de Manabí, que presentan períodos de expansión y contracción considerablemente más pronunciados. Asimismo, BP Citibank muestra episodios de crecimiento elevado en la cartera de créditos, mientras que BP Bolivariano presenta fluctuaciones importantes en determinados períodos. Los casos más extremos se observan en el crecimiento de cartera y activos, con tasas que superan el 100% en algunas entidades, y en el crecimiento patrimonial, donde se presentan variaciones tanto fuertemente positivas como negativas. En conjunto, estos resultados permiten identificar diferentes patrones de estabilidad y volatilidad financiera entre las entidades, constituyendo una característica relevante para la posterior reducción dimensional y conformación de perfiles mediante PCA y clustering.

#### Heatmaps

In [ ]:
eda.generar_heatmaps(base_final, indicadores_balance, indicadores_notebook=indicadores_notebook, carpeta=RAIZ_PROYECTO / "images")

Los heatmaps muestran que BP Guayaquil, BP Pichincha, BP Produbanco, BP Internacional, BP Loja y BP Machala presentan patrones relativamente estables, con variaciones de crecimiento moderadas y sin cambios extremos prolongados. En contraste, BP Capital, BP D-MIRO S.A./Banco Atlántida S.A. y BP Banco Comercial de Manabí presentan mayores fluctuaciones, evidenciadas por cambios marcados en la intensidad de los valores a través del tiempo. En el crecimiento de la cartera de créditos destacan BP Capital y BP D-MIRO, que alcanzan períodos superiores al 100%, mientras que en el crecimiento de activos sobresalen BP Banco Comercial de Manabí y BP D-MIRO, con episodios de expansión considerable. Por su parte, el crecimiento patrimonial presenta los cambios más extremos en BP Capital, con valores superiores al 100%, y en BP FINCA S.A./Banco Amibank S.A., que registra períodos de contracción significativa. En conjunto, los heatmaps permiten identificar entidades con trayectorias de crecimiento relativamente estables frente a otras con mayor volatilidad y cambios estructurales, aportando evidencia de la existencia de perfiles financieros diferenciados que posteriormente podrán ser caracterizados mediante las técnicas de reducción dimensional y clustering.

#### Diagramas de caja (Boxplots)

In [ ]:
eda.generar_boxplots(base_final, indicadores_balance, indicadores_notebook=indicadores_notebook, carpeta=RAIZ_PROYECTO / "images")

Los diagramas de caja evidencian una elevada dispersión de las variables de crecimiento interanual y una importante presencia de valores atípicos. En VAR_CARTERA DE CRÉDITOS, se observa una concentración considerable de atípicos por encima del límite superior, con valores que superan el 100%, así como observaciones inferiores a −50%, lo que refleja períodos de expansión y contracción muy pronunciados en determinadas entidades. En VAR_TOTAL ACTIVO, los atípicos superiores alcanzan aproximadamente 85%, mientras que los inferiores llegan alrededor de −33%, evidenciando también episodios de crecimiento y reducción significativos. Finalmente, VAR_TOTAL PATRIMONIO presenta una dispersión particularmente elevada, con valores superiores al 100% y observaciones inferiores a −75%. En conjunto, los resultados muestran que, aunque la mayor parte de las observaciones se concentra en rangos moderados de crecimiento, existen entidades y períodos con comportamientos financieros considerablemente más extremos. Estos valores serán conservados para las etapas posteriores, previa validación de su consistencia, debido a que pueden representar características reales de expansión o contracción y contribuir a la diferenciación de perfiles financieros mediante el PCA y el clustering.

#### Identificación de atípicos por el método del rango intercuartílico

In [ ]:
tabla_atipicos = eda.detectar_atipicos_iqr(
    base_final,
    indicadores_balance
)

In [ ]:
conteo_atipicos = pd.pivot_table(
    tabla_atipicos,
    index=["ENTIDAD", "INDICADOR"],
    columns="TIPO_ATIPICO",
    values="FECHA",
    aggfunc="count",
    fill_value=0
).reset_index()

# Total de atípicos
conteo_atipicos["TOTAL_ATIPICOS"] = (
    conteo_atipicos.get("Inferior", 0)
    + conteo_atipicos.get("Superior", 0)
)

# Ordenar de mayor a menor
conteo_atipicos = conteo_atipicos.sort_values(
    "TOTAL_ATIPICOS",
    ascending=False
).reset_index(drop=True)


In [ ]:
# Número de valores no nulos por entidad e indicador
total_valores = (
    base_final
    .groupby("ENTIDAD")[indicadores_balance]
    .count()
    .reset_index()
    .melt(
        id_vars="ENTIDAD",
        var_name="INDICADOR",
        value_name="TOTAL_VALORES"
    )
)

# Unir con tabla de atípicos
atipicos_f = conteo_atipicos.merge(
    total_valores,
    on=["ENTIDAD", "INDICADOR"],
    how="left"
)

#Porcentaje de valores atípicos problemáticos
atipicos_f["PORC_ATIPICOS"] = (
    atipicos_f["TOTAL_ATIPICOS"]
    / atipicos_f["TOTAL_VALORES"]
    * 100
)

In [ ]:
atipicos_f

In [ ]:
atipicos_f = atipicos_f.sort_values(
    "PORC_ATIPICOS",
    ascending=False
)

In [ ]:
atipicos_f

La detección de valores atípicos mediante el criterio de $1,5\times IQR$ evidencia que las observaciones extremas no se distribuyen de manera homogénea entre las entidades. 

Los mayores niveles de atipicidad se concentran en instituciones que presentan cambios estructurales o dinámicas de crecimiento diferenciadas. Por ejemplo, Banco Amibank registra una elevada frecuencia de atípicos inferiores en el crecimiento patrimonial, comportamiento consistente con la reducción de su patrimonio y el posterior proceso de liquidación. 

D-MIRO S.A./Banco Atlántida presenta numerosos atípicos tanto superiores como inferiores en cartera, activos, depósitos y patrimonio, coherentes con la fuerte reducción de cartera registrada en 2024 y la posterior transformación y expansión de la entidad. 

Por su parte, Banco Capital y Banco Comercial de Manabí presentan una concentración importante de atípicos superiores asociada con períodos de expansión de activos, cartera y captaciones. 

En entidades como Diners y Citibank, los valores atípicos deben interpretarse considerando su estructura particular de negocio y de fondeo, ya que tasas de crecimiento elevadas pueden estar asociadas a variaciones sobre bases relativamente pequeñas. En consecuencia, los resultados obtenidos presentan coherencia con las trayectorias observadas en el análisis temporal y los heatmaps, por lo que los valores atípicos identificados no serán eliminados automáticamente. 

Serán considerados observaciones potencialmente informativas sobre cambios estructurales y perfiles diferenciados de las entidades, previa validación de su consistencia con las fuentes originales.

In [ ]:
log_eda

## Análisis de Correlación

El análisis de correlación permite evaluar la intensidad y dirección de la relación lineal entre las variables numéricas del conjunto de datos. En este proyecto, se utiliza para identificar indicadores que contienen información muy similar o redundante, especialmente cuando presentan correlaciones cercanas a 1 o -1.

Este análisis es relevante antes de aplicar el Análisis de Componentes Principales (PCA), debido a que la presencia de variables altamente correlacionadas puede generar redundancia en la información utilizada para representar las dimensiones del conjunto de datos. La identificación y tratamiento de estas relaciones permite construir una base con variables más diferenciadas, evitando conservar indicadores que representen esencialmente la misma característica financiera.

La correlación no implica causalidad; en este análisis se utiliza como criterio exploratorio para identificar relaciones lineales y apoyar las decisiones de selección de variables. Las variables no se eliminan únicamente por presentar una correlación elevada, sino que cada relación se revisa considerando también la definición financiera de los indicadores y la información que aporta cada variable.

In [ ]:
base_final.columns.to_list

In [ ]:
columnas_excluir = [
    "MES",
    "AÑO",
    "ENTIDAD",
    "MES_NUM",
    "FECHA"
]

variables_correlacion = [
    col for col in base_final.columns
    if col not in columnas_excluir
]

df_corr = base_final[variables_correlacion]

print(f"Número de variables: {len(variables_correlacion)}")
print(variables_correlacion)

### Matriz de correlación 

In [ ]:
# Calcular la correlación lineal de Pearson entre los indicadores financieros.
# r = 1  -> relación lineal positiva perfecta
# r = 0  -> ausencia de relación lineal
# r = -1 -> relación lineal negativa perfecta

corr = df_corr.corr(method="pearson")

In [ ]:
# Carpeta donde se almacenará la gráfica
carpeta = RAIZ_PROYECTO / "images"

# Crear la carpeta si no existe
carpeta.mkdir(parents=True, exist_ok=True)

# Ruta de salida de la matriz de correlación
ruta_salida = carpeta / "matriz_correlacion_indicadores_financieros.png"

plt.figure(figsize=(20, 16))

sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)

plt.title("Matriz de correlación de los indicadores financieros")
plt.tight_layout()

# Guardar la gráfica
plt.savefig(
    ruta_salida,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Matriz de correlación guardada en: {ruta_salida.resolve()}")

Se identifican las correlaciones más elevadas, por lo que se toma el triángulo superior de la matriz y así evitar analizar dos veces cada par de variables.

In [ ]:
# Tomar solo el triángulo superior
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)


# Convertir la matriz de correlaciones en una tabla de pares de variables.
correlaciones = (
    corr.where(mask)
        .stack()
        .reset_index()
)

# Renombrar las columnas para facilitar la interpretación.
correlaciones.columns = ["VARIABLE_1", "VARIABLE_2", "CORRELACION"]


# Se utiliza el valor absoluto porque interesa identificar
# la intensidad de la relación independientemente de su dirección.
correlaciones["ABS_CORRELACION"] = (
    correlaciones["CORRELACION"].abs()
)

correlaciones = correlaciones.sort_values(
    "ABS_CORRELACION",
    ascending=False
)

# --------------------------------------------------------
# Guardar resultados
# --------------------------------------------------------

ruta_results = RAIZ_PROYECTO / "results"
ruta_results.mkdir(parents=True, exist_ok=True)

ruta_correlaciones = ruta_results / "correlaciones.xlsx"

correlaciones.to_excel(
    ruta_correlaciones,
    index=False
)

print(f"Correlaciones guardadas en: {ruta_correlaciones.resolve()}")

# Mostrar las 20 correlaciones de mayor intensidad
correlaciones.head(20)

Se identificó una correlación perfecta $r=1,00$ entre MOROSIDAD DE LA CARTERA TOTAL y CARTERA IMPRODUCTIVA / CARTERA BRUTA. 

La revisión de la definición del indicador permitió determinar que ambas variables utilizan la misma expresión: cartera improductiva como numerador y cartera bruta como denominador, y aunque la morosidad mide  qué proporción de la cartera total presenta problemas de pago considerados como morosos, la segunda relación implica determinar la proporción de la cartera los créditos que ya no está generando normalmente ingresos financieros por intereses debido a su situación de deterioro o incumplimiento sobre el saldo de los créditos colocados por la entidad antes de descontar las provisiones para créditos incobrables, lo que significa la proporción de la cartera que tiene problemas de pago.
Por tanto, ambos indicadores representan el mismo concepto financiero y contienen información redundante. 

En consecuencia, se decidió excluir CARTERA IMPRODUCTIVA / CARTERA BRUTA del conjunto de variables que será utilizado en el análisis de reducción de dimensionalidad, conservando MOROSIDAD DE LA CARTERA TOTAL como indicador representativo de este fenómeno.


In [ ]:
print("Columnas disponibles:")
print(base_final.columns.tolist())

print("\n¿Existe CARTERA VENCIDA?")
print("CARTERA VENCIDA / CARTERA BRUTA" in base_final.columns)

print("\n¿Existe CARTERA QUE NO DEVENGA INTERESES?")
print("CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA" in base_final.columns)

***Primera eliminación por correlación perfecta positiva***

In [ ]:
# Eliminar el indicador debido a que presenta correlación perfecta
# y representa información redundante respecto a MOROSIDAD DE LA CARTERA TOTAL.
base_final = base_final.drop(columns=["CARTERA IMPRODUCTIVA / CARTERA BRUTA"])

Se identificó una correlación negativa perfecta $r=-1,00$ entre ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS y ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS. Esta relación resulta razonable tanto desde el punto de vista matemático como financiero, debido a que ambos indicadores utilizan como denominador el total de activos y representan componentes complementarios de la estructura de activos de la entidad. 
Activos productivos / Total activos mide qué proporción de los activos de la entidad corresponde a activos productivos, es decir, activos que contribuyen a generar ingresos financieros.
Mientras que Activos improductivos netos / Total activos mide qué proporción de los activos corresponde a activos improductivos netos, es decir, aquellos activos que no generan el rendimiento financiero esperado, considerando el efecto de las provisiones correspondientes.

En consecuencia, cuando aumenta la proporción de activos productivos respecto del total de activos, disminuye proporcionalmente la participación de los activos improductivos netos, y viceversa. Por tanto, la correlación negativa perfecta indica que ambas variables contienen información esencialmente complementaria y redundante para el análisis multivariado. En consecuencia, se decidió excluir ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS y conservar ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS, evitando incorporar dos variables que representan la misma dimensión de la estructura de activos con relación inversa perfecta.

***Segunda eliminación por correlación perfecta negativa***

In [ ]:
# Eliminar ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS debido a su
# correlación negativa perfecta con ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS.
base_final = base_final.drop(columns=["ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS"])

Adicionalmente, se identificaron correlaciones elevadas entre la MOROSIDAD DE LA CARTERA TOTAL y los indicadores derivados de la composición de la cartera. En particular, se obtuvo una correlación de $r=0,973$ entre la morosidad de la cartera total y la proporción de CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA, mientras que la relación con CARTERA VENCIDA / CARTERA BRUTA presentó una correlación de $r=0,927$. Asimismo, los indicadores derivados de cartera improductiva presentaron correlaciones de $r=0,973$ y $r=0,927$ con las proporciones de cartera que no devenga intereses y cartera vencida, respectivamente. 

Además, ya se había identificado que $$ \text{Cartera improductiva} = \text{Cartera vencida} + \text{Cartera que no devenga intereses}, $$ 

y que la morosidad de cartera total está construida precisamente sobre la cartera improductiva respecto de la cartera bruta. Por eso, CARTERA IMPRODUCTIVA / CARTERA BRUTA es esencialmente la misma información que MOROSIDAD DE LA CARTERA TOTAL

Estas relaciones evidencian una elevada redundancia informativa entre las variables, debido a que la cartera improductiva está conformada por la cartera vencida y la cartera que no devenga intereses. Considerando además que la MOROSIDAD DE LA CARTERA TOTAL constituye el indicador oficial publicado por la Superintendencia de Bancos para representar el deterioro de la cartera total, se decidió conservar esta variable y excluir  CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA y CARTERA VENCIDA / CARTERA BRUTA, con el propósito de reducir la redundancia entre variables antes de aplicar las técnicas de reducción de dimensionalidad.

***Tercera eliminación por redundancia***

In [ ]:
# Indicadores que presentan elevada redundancia con
# MOROSIDAD DE LA CARTERA TOTAL.

a_eliminar=['CARTERA VENCIDA / CARTERA BRUTA',
       'CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA']


base_final = base_final.drop(columns=a_eliminar)

Existe otra relación elevada que, pese a superar el umbral de \(r>0,90\)  no fue considerada suficientes para eliminar las variables involucradas. En primer lugar, GASTOS DE OPERACION ESTIMADOS / TOTAL ACTIVO PROMEDIO y GASTOS DE PERSONAL ESTIMADOS / ACTIVO PROMEDIO presentaron una correlación de \(r=0,902\). Aunque esta relación es elevada, es coherente, puesto que los gastos de personal constituyen un componente de los gastos de operación, ambos indicadores representan dimensiones diferentes de la eficiencia: el primero refleja la carga operativa global de la entidad, mientras que el segundo permite identificar específicamente el peso de los gastos de personal. Por ello, se conservaron ambas variables al considerar que aportan información complementaria.

En segundo lugar, los indicadores RESULTADOS DEL EJERCICIO / PATRIMONIO PROMEDIO (ROE) y RESULTADOS DEL EJERCICIO / ACTIVO PROMEDIO (ROA) presentaron una correlación de \(r=0,844\). Esta relación es consistente con el hecho de que ambos indicadores comparten el resultado del ejercicio como numerador y miden distintas perspectivas de la rentabilidad. Sin embargo, sus denominadores son diferentes: el ROE relaciona el resultado con el patrimonio, mientras que el ROA lo relaciona con los activos. En consecuencia, ambos indicadores pueden aportar información diferenciada sobre el desempeño financiero y la estructura de las entidades, por lo que no se consideraron variables redundantes y se mantuvieron para las etapas posteriores del análisis multivariado.

In [ ]:
columnas_repetitivas=["MES_NUM",	"FECHA"]
### se eliminan las columnas repetitvas para las fechas de los gráficos
base_final = base_final.drop(columns=columnas_repetitivas)

In [ ]:
log_eda["secciones"]["analisis_correlacion"] = {
    "metodo": "Correlación de Pearson",

    "objetivo": (
        "Identificar relaciones lineales elevadas y posibles "
        "redundancias entre los indicadores financieros antes "
        "de aplicar PCA."
    ),

    "filas_eliminadas": 0,

    "variables_eliminadas": [
        "CARTERA IMPRODUCTIVA / CARTERA BRUTA",
        "ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS",
        "CARTERA VENCIDA / CARTERA BRUTA",
        "CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA"
    ],

    "cantidad_variables_eliminadas": 4,

    "decisiones": {
        "CARTERA IMPRODUCTIVA / CARTERA BRUTA": {
            "correlacion": 1.00,
            "decision": "Eliminar",
            "justificacion": (
                "Presenta correlación perfecta con MOROSIDAD DE LA CARTERA TOTAL "
                "y representa información redundante."
            )
        },

        "ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS": {
            "correlacion": -1.00,
            "decision": "Eliminar",
            "justificacion": (
                "Presenta correlación negativa perfecta con ACTIVOS PRODUCTIVOS "
                "/ TOTAL ACTIVOS, por lo que ambas variables representan "
                "información complementaria y redundante."
            )
        },

        "CARTERA VENCIDA / CARTERA BRUTA": {
            "correlacion": 0.927,
            "decision": "Eliminar",
            "justificacion": (
                "Presenta una relación elevada con MOROSIDAD DE LA CARTERA TOTAL "
                "y contribuye a la redundancia entre indicadores de deterioro de cartera."
            )
        },

        "CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA": {
            "correlacion": 0.973,
            "decision": "Eliminar",
            "justificacion": (
                "Presenta una relación elevada con MOROSIDAD DE LA CARTERA TOTAL "
                "y contribuye a la redundancia entre indicadores de deterioro de cartera."
            )
        }
    },

    "variables_conservadas_por_informacion_complementaria": [
        "GASTOS DE OPERACION ESTIMADOS / TOTAL ACTIVO PROMEDIO",
        "GASTOS DE PERSONAL ESTIMADOS / ACTIVO PROMEDIO",
        "RESULTADOS DEL EJERCICIO / PATRIMONIO PROMEDIO",
        "RESULTADOS DEL EJERCICIO / ACTIVO PROMEDIO"
    ],

    "decision_general": (
        "Se eliminaron únicamente variables que presentaron redundancia "
        "informativa y cuya exclusión permitió conservar un conjunto de "
        "indicadores financieros más diferenciado para las etapas posteriores "
        "de PCA y clustering. Las variables con correlaciones elevadas pero "
        "que aportan información complementaria fueron conservadas."
    )
}

In [ ]:
log_eda

In [ ]:
log_eda["base_final"]={'filas': base_final.shape[0], 'columnas': base_final.shape[1]}

In [ ]:
log_eda

In [ ]:
columnas_excluir = [
    "MES",
    "AÑO",
    "ENTIDAD",
    "MES_NUM",
    "FECHA"
]

variables_correlacion = [
    col for col in base_final.columns
    if col not in columnas_excluir
]

df_corr = base_final[variables_correlacion]

print(f"Número de variables: {len(variables_correlacion)}")
print(variables_correlacion)

In [ ]:
# --------------------------------------------------------
# Guardar el log en la carpeta results
# --------------------------------------------------------

ruta_results = RAIZ_PROYECTO / "results"

# Se crea la carpeta results si no existe.
ruta_results.mkdir(
    parents=True,
    exist_ok=True
)

ruta_log = (
    RAIZ_PROYECTO
    / "results"
    / "eda_log2.json"
)

# Se guarda el registro en formato JSON.
with open(
    ruta_log,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        log_eda,
        archivo,
        indent=4,
        ensure_ascii=False
    )

print(f"Log guardado en: {ruta_log}")

In [ ]:
# Se construye la ruta de salida a partir de la raíz del proyecto.
ruta_salida = RAIZ_PROYECTO / "data_processed" / "base_pca2.xlsx"

# Se crea la carpeta de salida si aún no existe.
ruta_salida.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Se guarda la base final resultante del EDA que será utilizada
# como entrada para el análisis de componentes principales (PCA).
base_final.to_excel(
    ruta_salida,
    index=False
)

print(f"Base preprocesada guardada en: {ruta_salida}")

## Conclusiones

1. Se identificó una elevada heterogeneidad en el comportamiento financiero de las entidades bancarias analizadas. Los análisis de series temporales, heatmaps y diagramas de caja muestran diferencias persistentes entre instituciones en indicadores de solvencia, calidad de activos, riesgo de crédito, liquidez, rentabilidad y crecimiento. Esta heterogeneidad constituye evidencia exploratoria de la existencia de perfiles financieros diferenciados.
Los indicadores de solvencia, cobertura y liquidez presentan diferencias estructurales importantes entre las entidades. La suficiencia patrimonial muestra una dispersión considerable, con instituciones que mantienen niveles elevados y otras que presentan períodos inferiores al umbral regulatorio del 9%. Asimismo, la cobertura de la cartera problemática presenta una elevada variabilidad, mientras que el indicador de fondos disponibles respecto de los depósitos de corto plazo evidencia diferentes posiciones de liquidez.
Los indicadores relacionados con la calidad y el deterioro de la cartera evidencian diferencias significativas entre instituciones. La morosidad de la cartera total presenta niveles considerablemente superiores en determinadas entidades y, en algunos períodos, supera el 30%. De manera consistente, los indicadores de cartera improductiva y cartera que no devenga intereses muestran mayores niveles en entidades específicas, evidenciando que el riesgo de crédito y la calidad de cartera no se distribuyen homogéneamente.
2. Los valores atípicos identificados no fueron considerados automáticamente como errores de datos. El criterio del rango intercuartílico permitió detectar numerosas observaciones extremas, pero su persistencia en determinadas entidades y períodos sugiere que pueden estar asociadas con características estructurales, especialización de negocio o cambios relevantes en la situación financiera. Por esta razón, el EDA conservó los valores atípicos y los utilizó como información potencialmente relevante para diferenciar perfiles financieros.
3. Se observaron entidades con comportamientos financieros particularmente diferenciados. BP FINCA S.A./BANCO AMIBANK S.A. presenta niveles elevados de morosidad y de cartera improductiva, mientras que D-MIRO muestra también niveles elevados de deterioro y una importante variabilidad. Por otra parte, entidades como BP Capital y BP Banco Comercial de Manabí destacan por niveles elevados de activos improductivos y por variaciones importantes en determinados indicadores. Estos casos refuerzan la existencia de estructuras financieras heterogéneas.

4. La variabilidad de los indicadores de crecimiento debe interpretarse considerando la estructura y el tamaño de cada entidad. Los valores extremos no necesariamente representan un comportamiento financiero anómalo, ya que tasas elevadas pueden originarse en variaciones sobre bases relativamente pequeñas o en cambios estructurales de las instituciones. En consecuencia, el EDA decidió conservar estas observaciones, previa validación de su consistencia con las fuentes originales.
5. Se eliminaron tres indicadores de morosidad segmentada debido a problemas de comparabilidad entre entidades. Se excluyeron los indicadores correspondientes a consumo, inmobiliario y microcrédito, debido a que la especialización y composición de las carteras no es homogénea entre las instituciones. Se conservó la morosidad de la cartera total como medida agregada y más comparable del deterioro de la cartera.
6. El análisis de correlación permitió reducir la redundancia informativa entre variables. Se identificaron cuatro relaciones que justificaron la eliminación de indicadores: CARTERA IMPRODUCTIVA / CARTERA BRUTA presentó correlación perfecta positiva de 1,00 con la morosidad de cartera total; ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS presentó correlación perfecta negativa de −1,00 con ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS; además, CARTERA VENCIDA / CARTERA BRUTA y CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA presentaron correlaciones de 0,927 y 0,973, respectivamente, con la morosidad de cartera total.
La eliminación de variables se realizó considerando tanto criterios estadísticos como financieros. No se eliminaron variables únicamente por presentar una correlación elevada. Por ejemplo, se conservaron los indicadores de gastos de operación y gastos de personal, a pesar de su correlación de 0,902, debido a que representan dimensiones complementarias de la eficiencia. Asimismo, se conservaron ROE y ROA, cuya correlación de 0,844 refleja una relación esperable pero permite analizar la rentabilidad desde perspectivas diferentes.
6. El EDA permitió construir una base más diferenciada para las técnicas multivariadas posteriores. Las decisiones adoptadas redujeron indicadores redundantes sin eliminar las observaciones extremas que pueden contener información relevante sobre las características particulares de las entidades. 
7. Como resultado final del EDA, se obtuvo una base de 1.207 observaciones y 27 variables. Esta base constituye el insumo para la siguiente etapa del proyecto, en la que se aplicará reducción de dimensionalidad mediante PCA y posteriormente técnicas de clustering para identificar y caracterizar perfiles financieros diferenciados entre las entidades.